# RFdiffusion3 execution example
Direct RFdiffusion-v3 structure design model for backbone generation and binder scaffold design.
## Overview
- **Field** — Protein design
- **Category** — Protein Design
- **Model type** — Diffusion
- **Year** — 2025
- **License** — BSD-3-Clause
**Related papers**

- [De novo Design of All-atom Biomolecular Interactions with RFdiffusion3](https://www.biorxiv.org/content/10.1101/2025.09.18.676967v1) — Jasper Butcher et al., bioRxiv, 2025
- [Accelerating Biomolecular Modeling with AtomWorks and RF3](https://www.biorxiv.org/content/10.1101/2025.08.14.670328v1) — Nathaniel Corley et al., bioRxiv, 2025

## Practice objective

Generate a small length-40 protein backbone without target or ligand conditioning. This does not validate function or binding.

## Requirements

Run the public model image with udocker on an x86_64 Linux Colab runtime. No Docker daemon or platform sign-in is required.

**A GPU runtime is required.** The notebook stops before downloading if no GPU is assigned. Memory requirements depend on the model and inputs.

This image requires compute capability 8.0 or newer (for example L4/A100). T4 is not supported.

Compressed image: approximately 6.1 GiB. The preflight checks a 20.3 GiB free-space budget for download and extraction. Run one model notebook at a time.

Image: `ghcr.io/connects-scv/laidd-rfdiffusion-v3@sha256:b27d414f3687bc35513a01b85f57ca13fc12d7f3e66ccd0e501b28835c54cccc`

Platform catalog version: not recorded. This notebook uses the model and weights in the image digest above. Do not assume results match a platform running another version.

Runtime documentation: https://indigo-dc.github.io/udocker/user_manual.html

Installed environments, containers, and results may disappear when a session ends. Download the result ZIP.

## Procedure

Run the runtime definition, environment setup, inputs, inference, and result cells in order. The first setup downloads a large image and may take time.

## Runtime helper

In [ ]:
"""Shared runtime embedded in LAIDD Hub container-example notebooks.

Only setup() downloads software. Import, input preparation, and result validation
can be tested independently. Images are public and pinned by OCI manifest digest.
"""
import base64
import csv
import gzip
import hashlib
import io
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import sys
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from sysconfig import get_path


class ColabModelExample:
    def __init__(self, spec, assets, work_root=None, execution_mode="F3"):
        if execution_mode not in {"P1", "P2", "F3"}:
            raise ValueError("execution_mode must be P1, P2 or F3")
        self.spec = spec
        self.assets = assets
        self.mode = execution_mode
        root = Path(work_root or (Path('/content') if Path('/content').is_dir() else Path.cwd()))
        self.root = root / ("laidd_" + spec["slug"])
        self.root.mkdir(parents=True, exist_ok=True)
        self.run_dir = self.root / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
        self.input_dir = self.run_dir / 'input'
        self.output_dir = self.run_dir / 'output'
        self.input_dir.mkdir(parents=True)
        self.output_dir.mkdir()
        self.env = dict(os.environ)
        self.env['UDOCKER_DIR'] = str(self.root / 'udocker')
        self.udocker = [str(Path(get_path('scripts')) / 'udocker')]
        if hasattr(os, 'geteuid') and os.geteuid() == 0:
            self.udocker.append('--allow-root')
        # udocker 1.3.17 accepts a registry digest in its tag field (repo:sha256:...).
        # Docker's repo@sha256 form is not accepted for fully-qualified repos by this release.
        self.image = spec['image'] + ':' + spec['digest']
        self.container = 'laidd-' + spec['slug'] + '-' + spec['digest'].split(':')[1][:12]
        self.parameters = None
        self.ready = False
        self.completed = False
        self.gpu_info = None
        self.image_metadata = None

    def _call(self, args, timeout=1800, capture=False):
        return subprocess.run(args, check=True, env=self.env, timeout=timeout,
                              text=True, stdout=subprocess.PIPE if capture else None,
                              stderr=subprocess.PIPE if capture else None)

    def _container_exists(self):
        if not Path(self.udocker[0]).is_file():
            return False
        return subprocess.run(self.udocker + ['inspect', self.container], env=self.env,
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=30).returncode == 0

    def preflight(self, cached=False):
        if platform.system() != 'Linux' or platform.machine().lower() not in {'x86_64', 'amd64'}:
            raise RuntimeError('Use an x86_64 Linux runtime, such as Google Colab. macOS/ARM is not supported by these images.')
        required = (0 if cached else self.spec['compressed_bytes'] * 3) + 2 * 1024**3
        free = shutil.disk_usage(self.root).free
        print(f"Disk budget: {required / 1024**3:.1f} GiB; available: {free / 1024**3:.1f} GiB")
        if free < required:
            raise RuntimeError('Not enough free disk for compressed layers and image extraction. Use a runtime with more disk space.')
        if self.spec['gpu']:
            if not shutil.which('nvidia-smi'):
                raise RuntimeError('A GPU is required. Select a GPU runtime before downloading the image.')
            result = self._call(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader,nounits'], timeout=30, capture=True)
            self.gpu_info = result.stdout.strip()
            print('GPU:', self.gpu_info)
            device = next(csv.reader(io.StringIO(self.gpu_info)))
            capability = float(device[2])
            if capability < self.spec.get('min_compute_capability', 0):
                raise RuntimeError('This image requires an Ampere-or-newer GPU (compute capability >= 8.0), such as L4/A100. T4 is not supported.')
        return {'required_disk_bytes': required, 'free_disk_bytes': free, 'gpu': self.gpu_info}

    def setup(self):
        self.preflight(cached=self._container_exists())
        self._call([sys.executable, '-m', 'pip', 'install', 'udocker==1.3.17'])
        self._call(self.udocker + ['install'])
        exists = self._container_exists()
        if not exists:
            print('Downloading pinned image:', self.spec['image'] + '@' + self.spec['digest'], flush=True)
            self._call(self.udocker + ['pull', '--platform=linux/amd64', self.image], timeout=5400)
            self._call(self.udocker + ['create', '--name=' + self.container, self.image], timeout=5400)
        self._call(self.udocker + ['setup', '--execmode=' + self.mode, self.container], timeout=600)
        if self.spec['gpu']:
            self._call(self.udocker + ['setup', '--nvidia', self.container], timeout=600)
            if self.mode.startswith('F'):
                self._call(self.udocker + ['setup', '--execmode=' + self.mode, '--force', self.container], timeout=600)
            probe = "import ctypes; c=ctypes.CDLL('libcuda.so.1'); assert c.cuInit(0)==0, 'CUDA driver init failed'; n=ctypes.c_int(); assert c.cuDeviceGetCount(ctypes.byref(n))==0 and n.value>0, 'GPU is not visible inside the container'; print('Container CUDA devices:', n.value)"
            self._call(self.command(argv=['python3', '-c', probe]), timeout=180)
        self.image_metadata = self._call(self.udocker + ['inspect', self.container], capture=True).stdout
        self.ready = True
        print('Environment ready. Input files:', self.input_dir)

    @staticmethod
    def _sequence(value):
        sequence = re.sub(r'\s+', '', str(value)).upper()
        if not sequence or re.search('[^ACDEFGHIKLMNPQRSTVWY]', sequence):
            raise ValueError('Use a non-empty amino acid sequence containing standard one-letter codes.')
        return sequence

    def prepare_inputs(self, parameters):
        p = dict(parameters)
        for filename, key in self.spec['assets'].items():
            asset = self.assets[key]
            raw = gzip.decompress(base64.b64decode(asset['gzip_base64']))
            if hashlib.sha256(raw).hexdigest() != asset['sha256']:
                raise ValueError('Example asset checksum mismatch: ' + key)
            (self.input_dir / filename).write_bytes(raw)
        slug = self.spec['slug']
        if slug in {'anthroab', 'abodybuilder3', 'humatch'}:
            heavy, light = self._sequence(p['heavy']), self._sequence(p['light'])
            if slug == 'humatch':
                with (self.input_dir / 'antibodies.csv').open('w', newline='') as handle:
                    writer = csv.writer(handle); writer.writerow(['heavy', 'light']); writer.writerow([heavy, light])
            else:
                name = 'heavy_light.fasta' if slug == 'abodybuilder3' else 'antibodies.fasta'
                (self.input_dir / name).write_text(f'>example_VH\n{heavy}\n>example_VL\n{light}\n')
        elif slug == 'immuscope':
            peptide = self._sequence(p['peptide'])
            if not 9 <= len(peptide) <= 21 or not re.fullmatch(r'[A-Za-z0-9_]+', str(p['allele'])):
                raise ValueError('Use a 9–21 residue peptide and a valid allele identifier.')
            (self.input_dir / 'peptides.txt').write_text(peptide + ' ' + p['allele'] + '\n')
        elif slug == 'ligtmap':
            if '\n' in p['smiles'] or not p['smiles'].strip():
                raise ValueError('Enter one SMILES on a single line.')
            # This teaching example uses the human Kinase class.
            if p['target_class'] != 'Kinase':
                raise ValueError('This example uses target_class=Kinase.')
            (self.input_dir / 'input.smi').write_text(p['smiles'].strip() + '\n')
            (self.input_dir / 'target.lst').write_text('Kinase\n')
        elif slug in {'openfold3', 'protenix'}:
            sequence = self._sequence(p['sequence'])
            if len(sequence) > 150:
                raise ValueError('Use at most 150 residues for this small-runtime example.')
            value = ({'queries': {'ubiquitin_example': {'chains': [{'molecule_type': 'protein', 'chain_ids': ['A'], 'sequence': sequence}]}}}
                     if slug == 'openfold3' else [{'name': 'ubiquitin_example', 'sequences': [{'proteinChain': {'sequence': sequence, 'count': 1}}]}])
            (self.input_dir / ('query.json' if slug == 'openfold3' else 'input.json')).write_text(json.dumps(value, indent=2))
        elif slug == 'rfdiffusion-v3':
            length = p['length']
            if isinstance(length, bool) or not isinstance(length, int) or not 30 <= length <= 100:
                raise ValueError('Use a backbone length from 30 to 100 for this example.')
            (self.input_dir / 'design.json').write_text(json.dumps({'unconditional_example': {'length': f'{length}-{length}'}}))
        for filename in self.spec['required_inputs']:
            path = self.input_dir / filename
            if not path.is_file() or path.stat().st_size == 0:
                raise ValueError('Missing input: ' + filename)
        self.parameters = p
        print('Prepared inputs:', ', '.join(self.spec['required_inputs']))
        return self.input_dir

    def command(self, argv=None):
        return self.udocker + ['run', '--nobanner',
            '--volume=' + str(self.input_dir.resolve()) + ':/work/input',
            '--volume=' + str(self.output_dir.resolve()) + ':/work/output',
            '--env=OMP_NUM_THREADS=2', '--env=MKL_NUM_THREADS=2',
            self.container, *(self.spec['argv'] if argv is None else argv)]

    def run(self, timeout=3600):
        if not self.ready or self.parameters is None:
            raise RuntimeError('Run setup() and prepare_inputs() first.')
        self.completed = False
        log_path = self.run_dir / 'inference.log'
        started = time.monotonic()
        print('Running inference; log:', log_path, flush=True)
        with log_path.open('w') as writer, log_path.open('r') as reader:
            process = subprocess.Popen(self.command(), env=self.env, stdout=writer, stderr=subprocess.STDOUT, start_new_session=True)
            try:
                while process.poll() is None:
                    output = reader.read(8192)
                    if output:
                        print(output, end='', flush=True)
                    if time.monotonic() - started > timeout:
                        raise TimeoutError('Inference exceeded its time limit. See inference.log.')
                    time.sleep(1)
                print(reader.read(), end='', flush=True)
                if process.returncode:
                    raise subprocess.CalledProcessError(process.returncode, self.command())
            finally:
                if process.poll() is None:
                    try:
                        os.killpg(process.pid, signal.SIGTERM)
                    except ProcessLookupError:
                        pass
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        os.killpg(process.pid, signal.SIGKILL)
                        process.wait()
        self.completed = True
        return self.validate_results()

    @staticmethod
    def validate_file(path):
        name = path.name.lower()
        text = gzip.decompress(path.read_bytes()).decode() if name.endswith('.gz') else path.read_text()
        if name.endswith(('.sdf', '.sdf.gz')):
            if '$$$$' not in text or 'M  END' not in text:
                raise ValueError('Invalid SDF output: ' + str(path))
        elif name.endswith('.pdb'):
            if not any(line.startswith('ATOM  ') for line in text.splitlines()):
                raise ValueError('No atoms in predicted PDB: ' + str(path))
        elif name.endswith('.cif'):
            if '_atom_site.' not in text:
                raise ValueError('No atom_site in predicted mmCIF: ' + str(path))
        elif name.endswith(('.fa', '.fasta')):
            records = text.split('>')[1:]
            if not records or not any(''.join(r.splitlines()[1:]).strip() for r in records):
                raise ValueError('Empty predicted sequences: ' + str(path))
        elif name.endswith('.csv'):
            delimiter = ';' if text.splitlines()[0].count(';') > text.splitlines()[0].count(',') else ','
            rows = list(csv.reader(io.StringIO(text), delimiter=delimiter))
            if len(rows) < 2 or not rows[0] or not any(any(cell.strip() for cell in row) for row in rows[1:]):
                raise ValueError('Empty prediction CSV: ' + str(path))
        return True

    def validate_results(self):
        files = sorted({p for pattern in self.spec['result_globs'] for p in self.output_dir.glob(pattern) if p.is_file() and p.stat().st_size})
        if not files:
            raise RuntimeError('Prediction output was not created. Check inference.log; logs/configuration files alone are not results.')
        for path in files:
            self.validate_file(path)
        if self.spec['slug'] == 'ligtmap':
            status = self.output_dir / 'Output/status.txt'
            if not status.exists() or ':Fail' in status.read_text():
                raise RuntimeError('LigTMap reported a failed calculation; see Output/status.txt.')
        return files

    def archive(self):
        if not self.completed:
            raise RuntimeError('Inference did not complete successfully.')
        files = self.validate_results()
        manifest = {'model': self.spec['slug'], 'image': self.spec['image'] + '@' + self.spec['digest'],
            'gpu': self.gpu_info, 'execution_mode': self.mode, 'host_python': platform.python_version(),
            'parameters': self.parameters, 'argv': self.spec['argv'], 'udocker_version': '1.3.17',
            'prepublication_colab_validation': 'not_performed',
            'inputs': [{'path': name, 'bytes': (self.input_dir / name).stat().st_size, 'sha256': hashlib.sha256((self.input_dir / name).read_bytes()).hexdigest()} for name in self.spec['required_inputs']],
            'reference_assets': {name: {'source': self.assets[key]['source'], 'sha256': self.assets[key]['sha256']} for name, key in self.spec['assets'].items()},
            'outputs': [{'path': str(p.relative_to(self.output_dir)), 'bytes': p.stat().st_size, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in files]}
        (self.run_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
        (self.run_dir / 'image_metadata.json').write_text(self.image_metadata or '{}')
        archive = self.run_dir / (self.spec['slug'] + '_results.zip')
        with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
            for path in self.output_dir.rglob('*'):
                if path.is_file():
                    bundle.write(path, str(Path('output') / path.relative_to(self.output_dir)))
            for name in ['run_manifest.json', 'image_metadata.json', 'inference.log']:
                bundle.write(self.run_dir / name, name)
        print('Validated prediction files:', [str(p.relative_to(self.output_dir)) for p in files])
        print('Result archive:', archive)
        return archive


In [ ]:
SPEC = json.loads("{\"slug\":\"rfdiffusion-v3\",\"image\":\"ghcr.io/connects-scv/laidd-rfdiffusion-v3\",\"digest\":\"sha256:b27d414f3687bc35513a01b85f57ca13fc12d7f3e66ccd0e501b28835c54cccc\",\"compressed_bytes\":6547018645,\"gpu\":true,\"min_compute_capability\":8,\"argv\":[\"rfd3\",\"design\",\"inputs=/work/input/design.json\",\"out_dir=/work/output\",\"n_batches=1\",\"diffusion_batch_size=1\"],\"parameters\":{\"length\":40},\"assets\":{},\"required_inputs\":[\"design.json\"],\"result_globs\":[\"**/*.cif\",\"**/*.pdb\"]}")
EXAMPLE_ASSETS = json.loads("{}")
EXECUTION_MODE = "F3"  # @param ["F3", "P1", "P2"]
example = ColabModelExample(SPEC, EXAMPLE_ASSETS, execution_mode=EXECUTION_MODE)
print('Model:', SPEC['slug'])
print('Input directory:', example.input_dir)
print('Output directory:', example.output_dir)

## Set up the environment

Check disk/GPU requirements, then prepare the pinned runtime and model image.

In [ ]:
example.setup()

## Set the inputs

Edit PARAMETERS below. To use another structure file, run this cell, replace the corresponding file in the printed input directory, and continue from the inference cell.

Required files: `design.json`

In [ ]:
PARAMETERS = json.loads("{\"length\":40}")
example.prepare_inputs(PARAMETERS)

## Run

Run the actual inference command. Failures and timeouts stop execution. Every run uses a new folder to keep previous results separate.

In [ ]:
prediction_files = example.run(timeout=3600)

## Check and download results

Validate structure, sequence, or CSV content, then archive outputs, logs, image identity, and input provenance.

In [ ]:
archive = example.archive()
for path in prediction_files:
    print(path.relative_to(example.output_dir), path.stat().st_size, 'bytes')
from IPython.display import display, FileLink
display(FileLink(str(archive)))
DOWNLOAD_RESULTS = True  # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print('Local runtime: download the ZIP using the link above.')
    else:
        files.download(str(archive))

## Interpreting results

Generate a small length-40 protein backbone without target or ligand conditioning. This does not validate function or binding.

Structures and scores are predictions. Consult the paper and model documentation for metric definitions. A successful small example does not reproduce the full paper or validate every platform input.